# Ollama + permanent tunnel (Colab / Kaggle)

Serves Ollama from this notebook's GPU and exposes it at **one address that
never changes**, so nothing has to be copied into a `.env` or `.dev.vars` after
the first time.

### How that works

`cloudflared tunnel --url ...` opens a *quick* tunnel, and a quick tunnel is
anonymous by design: Cloudflare hands out a throwaway `*.trycloudflare.com`
name on every start and takes it back when the process stops. That is why the
URL kept changing. The fix is a small Cloudflare Worker that holds the stable
name and forwards to whatever tunnel is live right now:

```
your app  ->  leadpulse-llm.<...>.workers.dev  ->  <random>.trycloudflare.com  ->  ollama :11434
              permanent                            new every session
```

This notebook tells that Worker where it moved to (Step 4) and keeps watch
(Step 5). The Worker lives in the `leadpulse` repo under `gateway/`.

### What is permanent and what is not

The **address** is permanent. The **uptime** is not: Colab and Kaggle stop a
session after a few hours and when idle, and when this notebook stops the model
is gone until you run it again. Nothing free changes that. What this removes is
the copy-paste and the stale-config failures.

### Before the first run

Enable the GPU, enable internet access, and store two secrets.

| | Colab | Kaggle |
| --- | --- | --- |
| GPU | Runtime -> Change runtime type -> T4 GPU | Settings -> Accelerator -> GPU T4 x2 |
| Internet | on by default | Settings -> Internet -> On |
| Secrets | key icon in the sidebar, "Notebook access" on | Add-ons -> Secrets |

**One secret is required**, the token:

| Secret | Value |
| --- | --- |
| `LLM_GATEWAY_TOKEN` | the Worker's `GATEWAY_TOKEN` — the same value as `LLM_API_KEY` in the app's `.dev.vars` |

The gateway address is already in Step 4 (`https://leadpulse-llm.20230905090503u.workers.dev`); it is not
a secret, since it is useless without the token. Set an optional
`LLM_GATEWAY_URL` secret only if the Worker ever moves.

On Kaggle, creating the secret is not enough — **attach it to this notebook** in
the same Add-ons → Secrets dialog, and turn Internet on. If Step 4 prompts you
for the token, that attachment is what is missing; typing it works, but only
for that session.

## Step 0 - check the runtime has a GPU

Run this first. On a CPU runtime a 7B model generates at roughly 2 tokens/second,
so every retrieval-sized prompt outlives Cloudflare's 100-second limit on time
to first byte and the app fails with `APIConnectionError ... 524`.

In [ ]:
# Fail fast on a CPU-only runtime instead of pulling several GB of models that
# cannot then serve a request in time.
import subprocess

try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    print("GPU detected:")
    for line in smi.stdout.strip().splitlines():
        print("  " + line)
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit(
        "No GPU on this runtime.\n"
        "  Colab:  Runtime -> Change runtime type -> T4 GPU\n"
        "  Kaggle: Settings -> Accelerator -> GPU T4 x2\n"
        "then re-run this notebook from the top."
    )

## Step 1 - install Ollama and cloudflared

In [ ]:
# zstd is a prerequisite of the Ollama installer.
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.ai/install.sh | sh

# -nc so a re-run of this cell does not re-download the binary.
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("Ollama and cloudflared installed.")

## Step 2 - start the Ollama daemon

`OLLAMA_HOST=0.0.0.0:11434` is what lets cloudflared reach it from outside
localhost; `OLLAMA_ORIGINS=*` stops the 403 that browser-origin requests
otherwise get.

In [ ]:
import os, subprocess, time

# Clear anything left over from an earlier run of this notebook.
!pkill -f ollama || true
time.sleep(1)

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

# Never evict the model. Ollama's default is to unload after 5 idle minutes,
# and reloading 20 GB takes minutes - during which the request produces no
# bytes at all and Cloudflare kills it at 100 seconds with a 524. An agent used
# in bursts would hit that on most first questions. This is the single setting
# that matters most for a large model behind a tunnel.
env["OLLAMA_KEEP_ALIVE"] = "-1"

ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(3)
print("Ollama daemon started.")

## Step 3 - pull the models

**Pick a model the GPU can actually hold.** `LLM_MODEL` in the app's
`.dev.vars` has to name one of these exactly, or every call 404s on the model
name.

| Runtime | VRAM | Largest sensible Qwen2.5 |
| --- | --- | --- |
| Colab free T4 | 16 GB | `qwen2.5:7b`, or `qwen2.5:14b` at q4 |
| Kaggle T4 x2 | 2 x 16 GB | `qwen2.5:32b` fits at q4 across both cards |
| Kaggle P100 | 16 GB | `qwen2.5:7b` (single card, no splitting) |

`qwen2.5:32b` at q4 is about 20 GB, so it does **not** fit a single T4 - on
Colab it will spill to CPU and crawl. That is the one mismatch to watch: the
`leadpulse` `.dev.vars` asks for `qwen2.5:32b`, which needs the Kaggle dual-T4
runtime.

In [ ]:
# Edit this list, then make LLM_MODEL in the app match one of the chat models.
MODELS = [
    "qwen2.5:7b",        # Colab T4. For Kaggle T4 x2, "qwen2.5:32b" fits at q4.
    "nomic-embed-text",  # embeddings; small, keep it.
]

for model in MODELS:
    print(f"\n--- pulling {model} ---")
    !ollama pull {model}

### Load the chat model into VRAM

Do this now, while you are watching, rather than letting the first real question
pay for it. Loading a 32B model takes minutes, and a request that produces no
byte for 100 seconds is killed by Cloudflare with a **524** — so a cold first
call fails even though everything is working. Step 2 set `OLLAMA_KEEP_ALIVE=-1`
so it stays loaded once it is in.

Expect several minutes here the first time, and a few seconds on a re-run.

In [ ]:
import json, time, urllib.request

CHAT_MODEL = MODELS[0]

print(f"loading {CHAT_MODEL} into VRAM - this is the slow one, sit tight...")
started = time.time()
request = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=json.dumps({"model": CHAT_MODEL, "prompt": "hi", "stream": False}).encode(),
    headers={"Content-Type": "application/json"},
)
# No Cloudflare in this path - it is a local call - so a long timeout is safe.
with urllib.request.urlopen(request, timeout=1800) as response:
    json.load(response)

print(f"{CHAT_MODEL} is resident after {time.time() - started:.0f}s.")
print("It will stay loaded for as long as this session lives.")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## Step 4 - open the tunnel and register it

Starts cloudflared, then tells the gateway Worker which random URL this session
got. Nothing printed here needs copying anywhere.

In [ ]:
import os, re, subprocess, threading, time
from queue import Empty, Queue

import requests

# cloudflared normally announces itself within a few seconds; 60 leaves room for
# a slow cold start without hanging the notebook forever.
URL_TIMEOUT_S = 60

# How often to re-register while nothing has changed. This doubles as a
# liveness signal - the gateway reports how long ago it last heard from this
# notebook, which is how you tell "session ended" from "model is just slow".
# Ten minutes keeps it inside the KV free tier's daily write allowance.
HEARTBEAT_S = 600

TUNNEL_CMD = [
    "./cloudflared", "tunnel",
    "--url", "http://localhost:11434",
    # Without this, Ollama sees a Host header it does not recognise and answers
    # 403 to every request as DNS-rebinding protection.
    "--http-host-header", "localhost:11434",
]


# The gateway's address is not a secret - it is useless to anyone without the
# token - so it is baked in here and only the token has to be stored. Override
# it with an LLM_GATEWAY_URL secret if the Worker is ever redeployed elsewhere.
DEFAULT_GATEWAY_URL = "https://leadpulse-llm.20230905090503u.workers.dev"


def _secret(name, default="", masked=True):
    """Reads a notebook secret: Colab, then Kaggle, then env, then a prompt.

    Falling all the way through to the prompt means the secret is not set up.
    On Kaggle, creating one is not enough - it has to be attached to this
    notebook under Add-ons -> Secrets, and the notebook needs Internet on.
    Typing it at the prompt works, but only for this session.
    """
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, "").strip()
    if value:
        return value
    if default:
        return default
    if not masked:
        return input(f"{name}: ").strip()
    from getpass import getpass
    print(f"{name} is not stored - set it under Add-ons -> Secrets (Kaggle) or the")
    print("key icon in the sidebar (Colab) to stop being asked every session.")
    return getpass(f"{name}: ").strip()


GATEWAY_URL = _secret("LLM_GATEWAY_URL", default=DEFAULT_GATEWAY_URL, masked=False).rstrip("/")
GATEWAY_TOKEN = _secret("LLM_GATEWAY_TOKEN")

# The User-Agent is set deliberately. Cloudflare's edge bot rules reject some
# default library agents with a 403 before the Worker is ever reached -
# `Python-urllib/3` is blocked today and `python-requests` is not, which is not
# a distinction worth betting this notebook on.
AUTH = {
    "Authorization": f"Bearer {GATEWAY_TOKEN}",
    "User-Agent": "leadpulse-notebook/1",
}

if not GATEWAY_URL or not GATEWAY_TOKEN:
    raise SystemExit("LLM_GATEWAY_URL and LLM_GATEWAY_TOKEN must both be set - see the top cell.")


def start_tunnel():
    """Starts cloudflared and returns it with the public URL it announced.

    The output is drained on a background thread rather than read inline: a pipe
    nobody reads fills its buffer and blocks the writer, which would stall
    cloudflared mid-session.
    """
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    time.sleep(1)

    proc = subprocess.Popen(
        TUNNEL_CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    found = Queue()

    def drain():
        pattern = re.compile(r"https://[a-zA-Z0-9][a-zA-Z0-9-]*\.trycloudflare\.com")
        announced = False
        for line in proc.stdout:
            if not announced:
                match = pattern.search(line)
                if match:
                    announced = True
                    found.put(match.group(0))
        # If cloudflared exits without printing a URL, unblock the waiter below
        # instead of making it sit out the whole timeout.
        if not announced:
            found.put("")

    threading.Thread(target=drain, daemon=True).start()

    try:
        url = found.get(timeout=URL_TIMEOUT_S)
    except Empty:
        proc.terminate()
        raise SystemExit("cloudflared never printed a tunnel URL - check the output above.")
    if not url:
        proc.terminate()
        raise SystemExit("cloudflared exited before opening a tunnel - check the output above.")
    return proc, url


def register(url, note=""):
    """Tells the gateway where the model is now."""
    res = requests.post(
        f"{GATEWAY_URL}/__register",
        json={"url": url, **({"note": note} if note else {})},
        headers=AUTH,
        timeout=20,
    )
    if res.status_code != 200:
        raise SystemExit(f"gateway refused the registration (HTTP {res.status_code}): {res.text}")


def self_test():
    """Checks the whole path - gateway, tunnel, Ollama - not just the tunnel."""
    res = requests.get(f"{GATEWAY_URL}/api/tags", headers=AUTH, timeout=30)
    res.raise_for_status()
    return [m.get("name") for m in res.json().get("models", [])]


proc, public_url = start_tunnel()
register(public_url, note="ollama notebook")

print("=" * 72)
print("Registered. This address is permanent - nothing to copy per session:")
print(f"  {GATEWAY_URL}")
print()
print("  leadpulse  .dev.vars :  LLM_BASE_URL=" + GATEWAY_URL)
print("  SupportBook .env     :  OLLAMA_API_BASE=" + GATEWAY_URL)
print()
print(f"(this session sits behind it at {public_url})")
print("=" * 72)

try:
    print("Self-test through the gateway:", self_test())
except Exception as exc:
    print(f"Self-test FAILED: {exc}")
    print("The tunnel is up but the gateway could not reach it. Check the token matches.")

## Step 5 - keep it alive

Leave this cell running: the tunnel is a child of this process, so stopping the
cell takes the model offline.

It also watches. A quick tunnel sometimes drops on its own well before the
session ends, and when it does `cloudflared` exits and the model goes
unreachable. Restarting it produces a *new* random URL - which is exactly what
the gateway absorbs, so this re-registers and the apps never learn anything
moved.

In [ ]:
print("Watching the tunnel. Leave this cell running.\n")

last_beat = time.time()
while True:
    time.sleep(15)

    if proc.poll() is not None:
        print(f"cloudflared exited (code {proc.returncode}); restarting it.")
        try:
            proc, public_url = start_tunnel()
            register(public_url, note="ollama notebook (restarted)")
            last_beat = time.time()
            print(f"  back up and re-registered at {public_url}")
            print("  callers may see 502 for up to a minute while the old address expires.")
        except SystemExit as exc:
            print(f"  could not restart the tunnel: {exc}")
            break

    elif time.time() - last_beat >= HEARTBEAT_S:
        try:
            register(public_url, note="ollama notebook (heartbeat)")
            last_beat = time.time()
        except SystemExit as exc:
            print(f"heartbeat failed: {exc}")

## Checking it from your machine

In the `leadpulse` repo:

```
npm run gateway:status
```

That reports each hop separately - gateway up, notebook registered and how long
ago, model reachable, and whether `LLM_MODEL` is among the models this notebook
actually pulled.

If the model stops answering, the age in that output is the thing to read: under
ten minutes means this notebook is alive and the model is merely slow; much
older means the session ended and this notebook needs running again.